**Note**: 

> This exercise has been written out in something called a Jupyter Notebook. We'll discuss Jupyter Notebooks in more detail later in this specialization—they are very a powerful tool for data science communication!—but for the time being, the notebook is just a convenient way for us to write out the exercise. You don't need to *do* anything with the notebook except read its contents—just use write your Python code in a regular `.py` file.

# Groupby and Arrest Data

In our merging exercises, we examined the relationship between county-level violent arrest totals and county-level drug arrest totals. In those exercises, you were given a dataset that provided you with county-level arrest totals. But that's not actually how the data are provided by the state of California. This week we will work with the *raw* California arrest data, which are not organized by county or even county-year. 

## Exercise 1: Learning the group structure of your data

**(1)** Load the data provided (Arrests.csv) which was downloaded from the California State Attorney General's office [here](https://openjustice.doj.ca.gov/data) (under "Arrests").

In [22]:
import pandas as pd
pd.set_option("mode.copy_on_write", True)

In [23]:
df=pd.read_csv("Arrests.csv")
df.head()

,YEAR,GENDER,RACE,AGE_GROUP,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL
0,1980,Male,Black,Under 18,Alameda County,505,1351,188,26,79,2149,2286,295
1,1980,Male,Black,18 to 19,Alameda County,205,465,183,8,48,909,1333,0
2,1980,Male,Black,20 to 29,Alameda County,949,1593,606,27,178,3353,7974,0
3,1980,Male,Black,30 to 39,Alameda County,450,755,241,18,110,1574,4876,0
4,1980,Male,Black,40 to 69,Alameda County,172,218,117,11,66,584,3836,0


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104651 entries, 0 to 104650
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   YEAR        104651 non-null  int64 
 1   GENDER      104651 non-null  object
 2   RACE        104651 non-null  object
 3   AGE_GROUP   104651 non-null  object
 4   COUNTY      104651 non-null  object
 5   VIOLENT     104651 non-null  int64 
 6   PROPERTY    104651 non-null  int64 
 7   F_DRUGOFF   104651 non-null  int64 
 8   F_SEXOFF    104651 non-null  int64 
 9   F_ALLOTHER  104651 non-null  int64 
 10  F_TOTAL     104651 non-null  int64 
 11  M_TOTAL     104651 non-null  int64 
 12  S_TOTAL     104651 non-null  int64 
dtypes: int64(9), object(4)
memory usage: 10.4+ MB


**(2)** What is the unit of observation for this dataset? In other words, when row zero says that there were 505 arrests for `VIOLENT` crimes, what exactly is that telling you -- 505 arrests in 1980? 505 arrests in Alameda County? Or is the data broken down even further?

In [25]:
#broken down by gender, race, age group, year, county

**(3)** Using what we discussed in the reading on "Checking for Duplicates", use `duplicated` to test if the variables *you* think uniquely identify rows in your data really do uniquely identify rows. If you were wrong, update your understanding of the data!

In [26]:
s=df[df.duplicated()]
s

,YEAR,GENDER,RACE,AGE_GROUP,COUNTY,VIOLENT,PROPERTY,F_DRUGOFF,F_SEXOFF,F_ALLOTHER,F_TOTAL,M_TOTAL,S_TOTAL


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104651 entries, 0 to 104650
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   YEAR        104651 non-null  int64 
 1   GENDER      104651 non-null  object
 2   RACE        104651 non-null  object
 3   AGE_GROUP   104651 non-null  object
 4   COUNTY      104651 non-null  object
 5   VIOLENT     104651 non-null  int64 
 6   PROPERTY    104651 non-null  int64 
 7   F_DRUGOFF   104651 non-null  int64 
 8   F_SEXOFF    104651 non-null  int64 
 9   F_ALLOTHER  104651 non-null  int64 
 10  F_TOTAL     104651 non-null  int64 
 11  M_TOTAL     104651 non-null  int64 
 12  S_TOTAL     104651 non-null  int64 
dtypes: int64(9), object(4)
memory usage: 10.4+ MB


**(4)** Once you have a handle on how the data are organized now, please **collapse the data** (using `groupby`) to be one observation per county-year-racial group. Make sure that you have the total arrest for violent crimes and the total arrests property crimes in your dataset.

In [28]:
df=df.groupby(["COUNTY","YEAR","RACE"],as_index=False)[["VIOLENT","PROPERTY"]].sum()
df.head()

,COUNTY,YEAR,RACE,VIOLENT,PROPERTY
0,Alameda County,1980,Black,2594,5138
1,Alameda County,1980,Hispanic,592,903
2,Alameda County,1980,Other,139,233
3,Alameda County,1980,White,1179,3103
4,Alameda County,1981,Black,2753,5533


**Note:** As mentioned in a previous reading, by default, `pandas` likes to make your grouping variables into a hierarchical index. Personally, I find hierarchical indices very weird and not worth dealing with. To avoid this, use the `as_index=False` option in `groupby`. I recommend this as a default option for working with `groupby`.

## Exercise 2: Investigating a question based on the data

In this section we'll work on the following question: does the racial composition of arrests in each county vary by arrest type? In other words, do Blacks make up a larger portion of the people arrested for drug offenses than violent offenses? To answer this question, you will need to compute the proportion of all arrests in a county-year that occur within each racial group. 

**(5)** We'll take this in steps. We want to get the fraction of violent crimes that correspond to each race and fo the same for property crimes AND we want to have these quantities for each county and each year. Before we can get fractions of a total, we need to calculate the total. Begin by creating a new column of data called "violent_total" that includes for each county and year sums violent crime arrests across all races. Hint: use `transform` to help you out.

You should be able to check your answer - 'violent_total' should be equal to the sum across all races for a given county and given year.

In [29]:
grouped=df.groupby(['COUNTY','YEAR'])
df['violent_total']=grouped['VIOLENT'].transform(lambda x:x.sum())
df.head()

,COUNTY,YEAR,RACE,VIOLENT,PROPERTY,violent_total
0,Alameda County,1980,Black,2594,5138,4504
1,Alameda County,1980,Hispanic,592,903,4504
2,Alameda County,1980,Other,139,233,4504
3,Alameda County,1980,White,1179,3103,4504
4,Alameda County,1981,Black,2753,5533,4699


**(6)** Repeat this process to create a new column of data called "property_total" that includes for each county and year sums property crime arrests across all races. Hint: once again use `transform` to help you out and check that "property_total" equals the sum of property crime arrests across all races for a given county and given year.

In [30]:
grouped=df.groupby(['COUNTY','YEAR'])
df['property_total']=grouped['PROPERTY'].transform(lambda x:x.sum())
df.head()

,COUNTY,YEAR,RACE,VIOLENT,PROPERTY,violent_total,property_total
0,Alameda County,1980,Black,2594,5138,4504,9377
1,Alameda County,1980,Hispanic,592,903,4504,9377
2,Alameda County,1980,Other,139,233,4504,9377
3,Alameda County,1980,White,1179,3103,4504,9377
4,Alameda County,1981,Black,2753,5533,4699,10014


## Exercise 3: Trying it on your own

**(7)** Now we have what we need to determine the fraction of violent and property crime arrests by race for each county and year. Add two new columns called "race_percent_violent" and "race_percent_property". To get the first, calculate each as the fraction of violent crime arrests by race divided by the total of violent crime arrests by race, multiplied by 100. Do a similar computation for "race_percent_property".

Always be sure to check your data to make sure the computation you were expecting worked - for example, percentages should add up to 100.

*A quick note of warning on interpretation: these results can tell you whether Black Californians make up a larger proportion of *arrests* for certain types of crimes, not whether they make up a larger proportion of people who *commit* a given type of crime! For example, there is extensive data that shows that Black and White Americans *use* drugs at the same rate, but Black Americans are arrested for drug use *much* more often. So be aware that arrests != crimes committed. Further analysis would be needed to explore those questions properly.*

In [65]:
#group_race=df.groupby('RACE')['VIOLENT'].sum()
#group_race
df["race_percent_violent"]=(df['VIOLENT']/df['violent_total'])*100
df["race_percent_property"]=(df['PROPERTY']/df['property_total'])*100
df.head()

,COUNTY,YEAR,RACE,VIOLENT,PROPERTY,violent_total,property_total,race_percent_violent,race_percent_property
0,Alameda County,1980,Black,2594,5138,4504,9377,57.593250,54.793644
1,Alameda County,1980,Hispanic,592,903,4504,9377,13.143872,9.629946
2,Alameda County,1980,Other,139,233,4504,9377,3.086146,2.484803
3,Alameda County,1980,White,1179,3103,4504,9377,26.176732,33.091607
4,Alameda County,1981,Black,2753,5533,4699,10014,58.586933,55.252646


**(8)** Let's say we wanted to identify the percentage of arrests by race for violent and property crime across ALL counties in California and compare that breakdown in 1980 to 2022. Repeat the analysis you completed above and modify it to gather this new set of information.

> **From your results, note the percent of violent crime associated with Black Californians in 1980 and in 2022. These values will be answers for your final quiz this week**

In [66]:
df1980=df.loc[(df['YEAR'])==1980]
df1980

,COUNTY,YEAR,RACE,VIOLENT,PROPERTY,violent_total,property_total,race_percent_violent,race_percent_property
0,Alameda County,1980,Black,2594,5138,4504,9377,57.593250,54.793644
1,Alameda County,1980,Hispanic,592,903,4504,9377,13.143872,9.629946
2,Alameda County,1980,Other,139,233,4504,9377,3.086146,2.484803
3,Alameda County,1980,White,1179,3103,4504,9377,26.176732,33.091607
172,Alpine County,1980,Other,0,0,1,9,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
9592,Yolo County,1980,White,125,407,197,585,63.451777,69.572650
9761,Yuba County,1980,Black,16,27,164,466,9.756098,5.793991
9762,Yuba County,1980,Hispanic,17,20,164,466,10.365854,4.291845
9763,Yuba County,1980,Other,3,7,164,466,1.829268,1.502146


In [89]:
group_race_1980p=df1980.groupby('RACE')['PROPERTY'].sum()
group_race_1980p

RACE
Black       48563
Hispanic    42282
Other        3738
White       83829
Name: PROPERTY, dtype: int64

In [90]:
group_race_1980v=df1980.groupby('RACE')['VIOLENT'].sum()
group_race_1980v

RACE
Black       29626
Hispanic    24605
Other        2327
White       29336
Name: VIOLENT, dtype: int64

In [70]:
df2022=df.loc[(df['YEAR'])==2022]
df2022

,COUNTY,YEAR,RACE,VIOLENT,PROPERTY,violent_total,property_total,race_percent_violent,race_percent_property
168,Alameda County,2022,Black,1618,1032,3485,2537,46.427547,40.677966
169,Alameda County,2022,Hispanic,941,790,3485,2537,27.001435,31.139141
170,Alameda County,2022,Other,408,185,3485,2537,11.707317,7.292077
171,Alameda County,2022,White,518,530,3485,2537,14.863702,20.890816
322,Alpine County,2022,Hispanic,0,0,4,1,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
9760,Yolo County,2022,White,119,107,345,232,34.492754,46.120690
9929,Yuba County,2022,Black,61,7,327,144,18.654434,4.861111
9930,Yuba County,2022,Hispanic,58,32,327,144,17.737003,22.222222
9931,Yuba County,2022,Other,23,15,327,144,7.033639,10.416667


In [91]:
group_race_2022v=df2022.groupby('RACE')['VIOLENT'].sum()
group_race_2022v

RACE
Black       22309
Hispanic    43864
Other        6979
White       24963
Name: VIOLENT, dtype: int64

In [92]:
group_race_2022p=df2022.groupby('RACE')['PROPERTY'].sum()
group_race_2022p

RACE
Black       11211
Hispanic    27975
Other        3737
White       19208
Name: PROPERTY, dtype: int64

In [76]:
df2022.groupby('RACE')['race_percent_violent'].mean()

RACE
Black       12.350753
Hispanic    33.443851
Other        9.293727
White       45.124612
Name: race_percent_violent, dtype: float64

In [72]:
df2022.groupby('RACE')['race_percent_property'].mean()

RACE
Black       11.120868
Hispanic    29.979188
Other        7.613048
White       51.478636
Name: race_percent_property, dtype: float64